# Prepare shared Stage 2 plans

Coordinator entry point. Read `docs/stage2_interfaces.md` at the published base.
Mount the intended Drive account in Colab before setup. This notebook does not mount,
authenticate to Hugging Face, install GPU packages, load weights, fit heads or run workers.
Install the CPU environment with `pip install -r requirements-stage2-cpu.txt` if needed;
record actual versions. Never update the checkout midway through a run.
Worker notebooks consume the printed plan; per-runtime Gemma authentication remains required
in model-loading notebooks. A passed plan describes readiness and inventory, not arrays.


## Colab setup
Run this setup first on a fresh runtime. It mounts Drive, obtains the integrated phase1 checkout, adds its import path and installs the appropriate dependency profile. Existing checkouts are never reset or updated automatically.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/content/Research')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO_URL = 'https://github.com/Drazr/Research.git'
BRANCH = 'phase1'
STORAGE_SCOPE = 'persistent-drive'

try:
    from google.colab import drive
except ImportError:
    if STORAGE_SCOPE != 'local-development':
        raise RuntimeError('Outside Colab, set explicit paths and local-development scope.')
else:
    drive.mount('/content/drive')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError('Mount the intended Drive containing the accepted phase1 artifacts.')
if not (REPO / '.git').is_dir():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError('Repository destination is nonempty; choose a clean checkout.')
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', '-c', 'core.autocrlf=false', 'clone', '--branch', BRANCH,
                    REPO_URL, str(REPO)], check=True)

def checkout_value(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()

if checkout_value('remote', 'get-url', 'origin').removesuffix('.git').rstrip('/') != REPO_URL.removesuffix('.git'):
    raise RuntimeError('Expected Drazr/Research checkout.')
if checkout_value('branch', '--show-current') != BRANCH or checkout_value('status', '--porcelain'):
    raise RuntimeError('Use a clean phase1 checkout; do not switch or pull during a running job.')
for filename in ('interfaces.py', 'extract_features.py', 'prepare_e03.py'):
    if not (REPO / 'src/stage2' / filename).is_file():
        raise RuntimeError('This checkout predates Stage 2 integration; update it before setup.')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ['PHASE1_ROOT'] = str(PHASE1_ROOT)
profile = REPO / 'requirements-stage2-cpu.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(profile)], check=True)
print('Frozen implementation commit:', checkout_value('rev-parse', 'HEAD'))
print('Persistent root:', PHASE1_ROOT)


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

if not (REPO / 'src/stage2/interfaces.py').is_file():
    raise ValueError('Set REPO to the shared-foundation checkout before continuing')
if not PHASE1_ROOT.is_dir():
    raise ValueError('Mount the intended Drive account and set PHASE1_ROOT')
os.chdir(REPO)
CODE_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
if subprocess.check_output(['git','status','--porcelain'], text=True).strip():
    raise ValueError('Use a clean frozen checkout; do not pull during a job')
print(json.dumps({'repo': str(REPO), 'phase1_root': str(PHASE1_ROOT), 'code_commit': CODE_COMMIT}, indent=2))


In [ ]:
# Rerun this cell to replace settings; execution always starts a fresh verifier process.
# First run: shared plan, with no model bindings, is sufficient for E03.
# Later runs: set SHARED_PLAN to that printed manifest to add model bindings.
SHARED_PLAN = None
DATASET_MANIFEST = PHASE1_ROOT / 'shared/8cf019d18a28a8bb18540f4ce2e26b17fc0af15a12c8d016138dc99c397c855a/manifests/54825d236efcd88cc5252918ebce6d1dad6109ef10a244281f262204bc6cc5bd/_COMPLETE.json'
DATA_VALIDATION = None  # Typed deterministic discovery in this dataset/protocol namespace.
BINDINGS = {}
# Optional, bind once here (no weights are loaded):
# BINDINGS = {'qwen': {'configuration_manifest': '/.../_COMPLETE.json',
#                      'model_validation': None, 'smoke': None}}
# None evidence pointers trigger typed discovery in the exact model namespace.
# Use 'gemma' for the other model. Never enter HF tokens in this mapping.


In [ ]:
import tempfile
from src.stage2.schema import marker_path
if SHARED_PLAN is not None:
    marker_path(SHARED_PLAN, PHASE1_ROOT)
else:
    marker_path(DATASET_MANIFEST, PHASE1_ROOT)
if DATA_VALIDATION is not None:
    marker_path(DATA_VALIDATION, PHASE1_ROOT)
for model, binding in BINDINGS.items():
    if model not in ('qwen','gemma'):
        raise ValueError('Unknown MODEL binding')
    for key, value in binding.items():
        if value is not None:
            p = marker_path(value, PHASE1_ROOT)
            if not p.is_file(): raise ValueError('Missing supplied binding: '+key)
print(json.dumps({'shared_plan':str(SHARED_PLAN), 'dataset_manifest':str(DATASET_MANIFEST),
                  'data_validation':str(DATA_VALIDATION), 'bindings':BINDINGS},indent=2))
command=[sys.executable,'-m','src.stage2.operations','prepare',
         '--phase1-root',str(PHASE1_ROOT),'--storage-scope',STORAGE_SCOPE]
if SHARED_PLAN is not None: command += ['--manifest',str(SHARED_PLAN)]
else: command += ['--dataset-manifest',str(DATASET_MANIFEST)]
if DATA_VALIDATION is not None: command += ['--data-validation',str(DATA_VALIDATION)]
with tempfile.TemporaryDirectory() as temporary:
    settings=Path(temporary)/'bindings.json'
    settings.write_text(json.dumps(BINDINGS),encoding='utf-8')
    completed=subprocess.run(command+['--bindings',str(settings)],text=True,capture_output=True)
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        raise RuntimeError('Stage 2 preparation blocked; inspect the diagnostic')
